In [36]:
#imports 

import pandas as pd 
import numpy as np 
from sklearn.metrics import balanced_accuracy_score

from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

In [37]:
#load data 
file_train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
file_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")


train_df = pd.DataFrame(file_train)
test_df = pd.DataFrame(file_test)


In [38]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [39]:
train_df.isnull().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [40]:
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(categorical_cols)

numerical_cols = train_df.select_dtypes(include=['float64']).columns.tolist()
print(numerical_cols)

['health_condition', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']


In [41]:
for col in numerical_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())


for col in categorical_cols: 
    train_df[col] = train_df[col].fillna("unknown")

In [42]:
train_df[:12]

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,unknown,average,sedentary,unknown,male
5,5,at-risk,5.11,82.8,24.43,2290.0,2134.0,29.2,2.02,veg,low,poor,sedentary,occasional,male
6,6,at-risk,8.21,71.1,24.14,2723.0,12984.0,54.5,2.01,balanced,low,unknown,moderate,no,female
7,7,at-risk,7.47,87.0,23.51,2241.0,3096.0,37.5,0.51,balanced,low,average,sedentary,occasional,male
8,8,unhealthy,5.94,75.4,24.36,2241.0,12165.0,61.6,1.56,veg,high,average,active,no,other
9,9,at-risk,6.97,80.1,20.47,2928.0,13108.0,51.9,2.53,veg,medium,good,active,no,male


### add the xgboosting model

In [43]:

le = LabelEncoder()
train_y = le.fit_transform(train_df['health_condition'])

train_x = train_df.drop(columns=["id", "health_condition"])
test_x = test_df.drop(columns=["id"])

categorical_cols = train_x.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_cols:
    train_x[col] = train_x[col].astype('category')
    test_x[col] = test_x[col].astype('category')

In [45]:

weights = compute_sample_weight(class_weight='balanced', y=y_tr)

X_tr, X_val, y_tr, y_val = train_test_split(
    train_x, 
    train_y, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_y
)


model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    enable_categorical=True,
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

model.fit(X_tr, y_tr, sample_weight=weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=3, ...)

In [46]:


val_probs = model.predict_proba(X_val)

class_priors = np.bincount(train_y) / len(train_y)

adjusted_probs = val_probs / class_priors

y_pred_adjusted = np.argmax(adjusted_probs, axis=1)

raw_score = balanced_accuracy_score(y_val, np.argmax(val_probs, axis=1))

print(f"Raw Argmax Score: {raw_score:.4f}")


Raw Argmax Score: 0.9489
Prior-Adjusted Score: 0.8959


In [47]:
test_predictions = model.predict(test_x)  

final_labels = le.inverse_transform(test_predictions)

submission = pd.DataFrame({
    'id': test_df['id'],
    'health_condition': final_labels
})
submission.to_csv('submission.csv', index=False)